# PhenoAge frente a los modelos de cada nivel

¿Ordena nuestro modelo quién muere antes igual o mejor que PhenoAge (Levine et al., 2018), la edad biológica de analítica más citada?

**PhenoAge** es una fórmula fija, sin nada que entrenar, con la edad y 9 análisis: albúmina, creatinina, glucosa, PCR, % de linfocitos, VCM, RDW, fosfatasa alcalina y leucocitos. Se construyó con NHANES III, así que en nuestros datos (NHANES 1999-2018) trabaja con personas que no ha visto. Nuestros modelos usan su riesgo de validación cruzada: cada persona, con un modelo entrenado sin ella.

**Con quién:** las personas de train con las 9 variables de PhenoAge. Sin PCR en 2011-2014, quedan fuera esos ciclos.

Listones fijados antes de ver los resultados:

- **Nivel 2** (analítica ampliada, como PhenoAge): no puede ser peor. Si el intervalo del bootstrap de la diferencia de C-index misma edad queda entero por debajo de 0, va a limitaciones y se revisa.
- **Nivel 1** (analítica básica): se acepta quedar algo por debajo, porque usa menos análisis; se mide la distancia.

Sin pesos muestrales. Todo dentro de train: **el test no se toca**.

Notebook de trabajo: importa de `src/`. Tiempo aproximado, si se ejecuta entero: **unos 45 minutos**, casi todo en los dos bootstraps.

In [1]:
# Imports
import numpy as np
import pandas as pd

from bioage.config.model_levels import ROUTINE, EXTRAS, QUESTIONNAIRE
from bioage.config.paths import GOLD_TABLE
from bioage.modeling.s1_split import split_data
from bioage.modeling.s4_biological_age import biological_age, fit_reference
from bioage.modeling.s5_evaluation import c_index, c_index_by_group, compare_risks, evaluate_level, same_age_c_index, weighted_medians
from bioage.utils.features import age_groups
from bioage.utils.notebook_helpers import print_split_summary
from bioage.config.phenoage import PHENOAGE_VARIABLES
from bioage.utils.phenoage import phenoage, phenoage_risk, phenoage_score

C:\Data\1_MisCosas\Evolve\TFM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Datos

Mismo reparto que en los niveles: 80/20 con 5 bloques dentro de train.

In [2]:
LEVELS = {
    "level_1": ROUTINE,
    "level_2": ROUTINE + EXTRAS,
    "level_3": ROUTINE + QUESTIONNAIRE,
    "level_4": ROUTINE + EXTRAS + QUESTIONNAIRE,
}

gold = pd.read_parquet(GOLD_TABLE)
train, test, folds = split_data(gold)
print_split_summary(train, test, folds)

complete = train[PHENOAGE_VARIABLES].notna().all(axis=1)
subset = train[complete]
print("People with every PhenoAge variable:", len(subset), "of", len(train), "| deaths:", int(subset["death"].sum()), "of", int(train["death"].sum()))
subset["cycle"].value_counts().sort_index()

          people       deaths      
part        test train   test train
cycle                              
1999_2000    829  3316    212   846
2001_2002    925  3697    188   750
2003_2004    877  3511    165   661
2005_2006    874  3496    120   479
2007_2008   1043  4170    136   544
2009_2010   1120  4479     99   396
2011_2012    993  3968     70   278
2013_2014   1061  4248     51   206
2015_2016   1011  4043     31   122
2017_2018    966  3867     14    57
train: 38795 people, 4339 deaths
test:  9699 people, 1086 deaths
folds: [7759, 7759, 7759, 7759, 7759] people in each validation part
People with every PhenoAge variable: 29946 of 38795 | deaths: 3745 of 4339


cycle
1999_2000    3239
2001_2002    3617
2003_2004    3444
2005_2006    3437
2007_2008    4104
2009_2010    4387
2015_2016    3956
2017_2018    3762
Name: count, dtype: int64

## 2. La fórmula de PhenoAge

Coeficientes y unidades del artículo original. Nuestras unidades son las del laboratorio español, así que se convierten antes:

| Variable | Nuestra unidad | PhenoAge | Conversión |
|---|---|---|---|
| Albúmina | g/dL | g/L | × 10 |
| Creatinina | mg/dL | µmol/L | × 88,4 |
| Glucosa | mg/dL | mmol/L | ÷ 18,016 |
| PCR | mg/L | mg/dL, en logaritmo | ÷ 10; la de 0 sube a 0,08 mg/L, el límite de detección que ya se usa |
| Linfocitos | recuento | % de los leucocitos | linfocitos ÷ leucocitos × 100 |

El resto (VCM, RDW, fosfatasa alcalina, leucocitos y edad) va en las mismas unidades.

- **`phenoage_score`:** puntuación de riesgo; ordena igual que el riesgo y es la que se usa para el C-index.
- **`phenoage_risk`:** riesgo de morir en 10 años según PhenoAge. En casos muy graves el ordenador lo redondea a 1, por eso el C-index usa la puntuación.
- **`phenoage`:** la edad a la que la población de NHANES III tiene ese riesgo, calculada desde la puntuación para no perder esos casos.

In [3]:
pheno_score = phenoage_score(subset)
pheno_risk = phenoage_risk(pheno_score)
pheno_age = phenoage(pheno_score)
print("People with PhenoAge risk rounded to 1:", int((pheno_risk == 1).sum()))
pd.DataFrame({"age": subset["age"], "phenoage": pheno_age, "gap": pheno_age - subset["age"]}).describe().T.round(2)

People with PhenoAge risk rounded to 1: 50


,count,mean,std,min,25%,50%,75%,max
age,29946.0,45.82,17.72,18.00,31.00,45.00,61.00,79.00
phenoage,29946.0,41.25,21.19,-2.22,24.24,40.24,56.83,311.53
gap,29946.0,-4.57,8.99,-24.58,-9.72,-6.17,-1.75,240.53


## 3. Riesgo de nuestros modelos

Cox simple de cada nivel, con validación cruzada sobre todo train, y después se toman las personas del subconjunto. PhenoAge entra con su puntuación. La edad biológica de cada nivel usa la referencia de cada nivel ajustada con todo train, como en el notebook de edad biológica.

In [4]:
oof_risks = {name: evaluate_level(columns, train, folds, models=["cox"])[1]["cox"]["risk"] for name, columns in LEVELS.items()}
references = {name: fit_reference(risk, train) for name, risk in oof_risks.items()}

risks = {"phenoage": pheno_score, **{name: risk.loc[subset.index] for name, risk in oof_risks.items()}}
ages = {"phenoage": pheno_age, **{name: pd.Series(biological_age(risks[name], subset["sex"], references[name]), index=subset.index) for name in LEVELS}}

## 4. C-index

Todos con las mismas personas y calculado de una vez sobre el riesgo de todos los bloques juntos (en los notebooks de nivel era la media de los 5 bloques), así que las cifras pueden diferir algo de las de esos notebooks.

| Columna | Qué mide |
|---|---|
| `c_index` | De dos personas, cuántas veces da más riesgo a la que muere antes en 10 años |
| `c_index_same_age` | Lo mismo, dentro de cada grupo de edad: **lo que más importa para el producto** |
| `c_index_female`, `c_index_male` | Lo mismo, por sexo |

In [5]:
def c_index_summary(risk):
    by_sex = c_index_by_group(risk, subset, subset["sex"])
    return {
        "c_index": c_index(risk, subset, subset),
        "c_index_same_age": same_age_c_index(c_index_by_group(risk, subset, age_groups(subset))),
        "c_index_female": by_sex.loc["female", "c_index"],
        "c_index_male": by_sex.loc["male", "c_index"],
    }

pd.DataFrame({name: c_index_summary(risk) for name, risk in risks.items()}).T.round(4)

,c_index,c_index_same_age,c_index_female,c_index_male
phenoage,0.8483,0.6899,0.8549,0.8423
level_1,0.8472,0.6875,0.8484,0.8410
level_2,0.8609,0.7219,0.8635,0.8544
level_3,0.8575,0.7180,0.8614,0.8500
level_4,0.8669,0.7393,0.8716,0.8593


Por grupo de edad: C-index dentro de cada grupo.

In [6]:
by_age = pd.DataFrame({name: c_index_by_group(risk, subset, age_groups(subset))["c_index"] for name, risk in risks.items()})
by_age["deaths"] = c_index_by_group(risks["phenoage"], subset, age_groups(subset))["deaths"].astype(int)
by_age.round(3)

,phenoage,level_1,level_2,level_3,level_4,deaths
18-29,0.602,0.626,0.678,0.633,0.666,55
30-39,0.703,0.683,0.715,0.707,0.731,63
40-49,0.717,0.689,0.751,0.737,0.772,173
50-59,0.709,0.716,0.755,0.754,0.776,306
60-69,0.708,0.705,0.735,0.740,0.756,673
70-79,0.671,0.671,0.702,0.695,0.716,1025


## 5. ¿Nivel 1 y nivel 2 frente a PhenoAge?

Bootstrap de la diferencia (nuestro modelo − PhenoAge), con las mismas personas en las dos, de unos 20 minutos cada uno. Positiva: ordena mejor el nuestro. La diferencia es real si el intervalo (`low`, `high`) no incluye el 0.

In [7]:
versus_level_1 = compare_risks(risks["level_1"], risks["phenoage"], subset)
versus_level_1

,difference,low,high
c_index,-0.0009,-0.0041,0.0016
c_index_same_age,-0.0020,-0.0107,0.0061


In [8]:
versus_level_2 = compare_risks(risks["level_2"], risks["phenoage"], subset)
versus_level_2

,difference,low,high
c_index,0.0127,0.0096,0.0158
c_index_same_age,0.0322,0.0224,0.0408


## 6. Calibración de la edad

Mediana de edad biológica − edad real por grupo de edad y sexo. Cerca de 0, la persona típica de cada grupo sale con su edad. PhenoAge se calibró con NHANES III (1988-1994); si la población de 1999-2018 está más sana a igual edad, saldrá más joven.

In [9]:
gaps = pd.DataFrame({name: age - subset["age"] for name, age in ages.items()})
weighted_medians(gaps, subset["weight_exam"], [age_groups(subset), subset["sex"]]).round(2)

phenoage  level_1  level_2  level_3  level_4
age   sex                                                 
18-29 female     -7.31    -0.27     0.10    -0.20     0.22
      male       -7.32    -0.07     0.15     0.20     0.62
30-39 female     -7.49     0.11    -0.08    -0.24    -0.08
      male       -6.71    -0.10     0.33     0.14     0.09
40-49 female     -7.20     0.57     0.65     0.28     0.09
      male       -6.37     0.00     0.25    -0.73    -0.30
50-59 female     -7.61    -0.68    -0.06    -0.93    -0.01
      male       -5.37     0.11     0.16    -0.05     0.22
60-69 female     -6.87    -0.87    -0.07    -0.71    -0.02
      male       -4.37    -0.26    -0.17    -0.33    -0.04
70-79 female     -5.86     0.43     0.85     0.45     0.59
      male       -3.10    -0.13     0.46    -0.18     0.42

## 7. ¿Se parecen?

Correlación persona a persona. Entre edades sale alta solo porque las dos llevan la edad; la que dice si miden lo mismo es la de la diferencia con la edad real.

In [10]:
pd.DataFrame({
    "age_correlation": {name: np.corrcoef(ages["phenoage"], ages[name])[0, 1] for name in LEVELS},
    "gap_correlation": {name: np.corrcoef(gaps["phenoage"], gaps[name])[0, 1] for name in LEVELS},
}).round(3)

,age_correlation,gap_correlation
level_1,0.934,0.541
level_2,0.928,0.538
level_3,0.920,0.491
level_4,0.919,0.513


## 8. Conclusión

**Con la analítica básica, el nivel 1 ordena igual que PhenoAge; con la ampliada, el nivel 2 la supera claramente.** Las cifras son de validación cruzada dentro de train, con las 29.948 personas que tienen las 9 variables de PhenoAge (sin 2011-2014); el test sigue sin tocar.

### Qué se hizo y por qué

**1. C-index con las mismas personas**

| | Variables | C-index | Misma edad |
|---|---|---|---|
| PhenoAge | 10 (con RDW y PCR) | 0,848 | 0,690 |
| Nivel 1 (Base) | 18 | 0,847 | 0,688 |
| Nivel 2 (Base + Extras) | 28 | 0,860 | 0,722 |
| Nivel 3 (Base + Cuestionario) | 22 | 0,857 | 0,718 |
| Nivel 4 (todo) | 32 | 0,866 | 0,739 |

El nivel 2 supera a PhenoAge en todos los grupos de edad; el nivel 1 queda por encima en 18-29 y 50-59, prácticamente igual en 60-79 y por debajo en 30-49.

**2. Bootstrap frente a PhenoAge (misma edad)**

- **Nivel 1:** −0,002 (intervalo del 95 %: −0,012 a 0,008). Incluye el 0: no se distingue de PhenoAge, sin RDW ni PCR y solo con lo que trae una analítica básica.
- **Nivel 2:** +0,033 (0,021 a 0,042). No incluye el 0: el listón se supera, unas 3 parejas más de cada 100 bien ordenadas entre personas de edad parecida.

**3. Calibración de la edad**

PhenoAge saca a la persona típica de cada grupo entre 3 y 7 años más joven que su edad, más en mujeres. Se calibró con NHANES III (1988-1994), y la población de 1999-2018 está más sana a igual edad. Es el desvío que corrige nuestra referencia, que deja la mediana de cada grupo dentro de ±1 año en todos los niveles.

**4. ¿Miden lo mismo?**

La correlación de la diferencia con la edad real entre PhenoAge y nuestros niveles es de 0,49 a 0,54: van en la misma dirección, pero no ponen a la misma gente en el mismo sitio.

### Resultado final

| Frente a PhenoAge | Resultado |
|---|---|
| Nivel 1 | Igual (diferencia no distinguible de 0) |
| Nivel 2 | Mejor (+0,033 misma edad) |
| Calibración | PhenoAge rejuvenece 3-7 años en esta población; nuestra referencia no |

### Pendiente

- Repetir la comparación en test, en la evaluación final.
- 50 personas tienen un riesgo de PhenoAge tan alto que el ordenador lo redondea a 1; su edad se calcula desde la puntuación y no cambia el C-index.